## Lab 2. ML on big data (Apache Spark, SparkML)
## Chast 1. Zadacha regressii: predskazanie total_amount (summa poezdki NYC Taxi)

#### Zapusk Spark-sessii

In [ ]:
import os
import csv
from pyspark.sql import SparkSession, DataFrame
from pyspark import SparkConf
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, MinMaxScaler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, CrossValidatorModel, ParamGridBuilder
from pyspark.ml import Pipeline
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType

In [ ]:
def create_spark_configuration() -> SparkConf:
    user_name = os.getenv("USER")
    conf = SparkConf()
    conf.setAppName("sobd lab2")
    conf.setMaster("yarn")
    conf.set("spark.submit.deployMode", "client")
    conf.set("spark.executor.memory", "12g")
    conf.set("spark.executor.cores", "8")
    conf.set("spark.executor.instances", "2")
    conf.set("spark.driver.memory", "4g")
    conf.set("spark.driver.cores", "2")
    conf.set("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.0")
    conf.set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    conf.set("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkCatalog")
    conf.set("spark.sql.catalog.spark_catalog.type", "hadoop")
    conf.set("spark.sql.catalog.spark_catalog.warehouse", f"hdfs:///user/{user_name}/warehouse")
    conf.set("spark.sql.catalog.spark_catalog.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")
    return conf

In [ ]:
conf = create_spark_configuration()

In [ ]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark

#### VARIANT: postav svoy nomer v gruppe mod 4. Model regressii podstavitsya avtomaticheski.
0,1 -> LinearRegression | 2 -> RandomForestRegressor | 3 -> GBTRegressor

In [ ]:
VARIANT = 0  # <<< POSTAV SVOY VARIANT (0,1,2,3)

def is_linear_reg(v): return v in (0, 1)

#### Zagruzka datataseta iz tablitsy, sozdannoy v LR1

In [ ]:
database_name = "shahulov_database"
table_name = "sobd_lab1_processed_table"  # dolzhno sovpadat s LR1
spark.catalog.setCurrentDatabase(database_name)
df = spark.table(table_name)
df.show()

In [ ]:
df.printSchema()

In [ ]:
df.count()

#### Postanovka zadachi
Predskazat total_amount (regressiya). Metriki: RMSE i R2.
Priznaki-summy (fare/tip/tolls) isklyucheny vo izbezhanie utechki.

In [ ]:
target_col = "total_amount"
categorical_features = ["PULocationID", "DOLocationID", "RatecodeID", "payment_type"]
numeric_features = ["trip_distance", "passenger_count", "trip_duration_min"]
for col_name in numeric_features + [target_col]:
    df = df.withColumn(col_name, col(col_name).cast(DoubleType()))

Otdelim ~1000 strok i sohranim v csv dlya sleduyushchey laby.

In [ ]:
def save_sample_to_csv(data, file_path, sample_size=1000):
    frac = sample_size / data.count()
    sample_data, remaining = data.randomSplit([frac, 1 - frac])
    with open(file_path, mode="w", newline="") as f:
        w = csv.writer(f)
        w.writerow(data.columns)
        for row in sample_data.take(sample_size):
            w.writerow(row)
    print(f"Saved {file_path}")
    return remaining

df = save_sample_to_csv(df, "streaming-data.csv", 1000)

In [ ]:
train_df, test_df = df.randomSplit([0.8, 0.2])
print(f"Train: {train_df.count()}, Test: {test_df.count()}")

#### Konveyer SparkML (kodirovka priznakov + model po variantu)

In [ ]:
def get_regressor(v, label):
    if is_linear_reg(v):
        return LinearRegression(featuresCol="features", labelCol=label, predictionCol="prediction", maxIter=20)
    if v == 2:
        return RandomForestRegressor(featuresCol="features", labelCol=label, predictionCol="prediction")
    return GBTRegressor(featuresCol="features", labelCol=label, predictionCol="prediction", maxIter=20)

def create_pipeline():
    idx = [f"{f}_idx" for f in categorical_features]
    si = StringIndexer(inputCols=categorical_features, outputCols=idx, handleInvalid="keep")
    if is_linear_reg(VARIANT):
        ohe_cols = [f"{f}_ohe" for f in categorical_features]
        ohe = OneHotEncoder(inputCols=idx, outputCols=ohe_cols, handleInvalid="keep")
        num_asm = VectorAssembler(inputCols=numeric_features, outputCol="num_vec")
        scaler = MinMaxScaler(inputCol="num_vec", outputCol="num_scaled")
        asm = VectorAssembler(inputCols=ohe_cols + ["num_scaled"], outputCol="features")
        stages = [si, ohe, num_asm, scaler, asm]
    else:
        asm = VectorAssembler(inputCols=idx + numeric_features, outputCol="features")
        stages = [si, asm]
    stages.append(get_regressor(VARIANT, target_col))
    return Pipeline(stages=stages)

pipeline = create_pipeline()

#### Podbor giperparametrov (kross-validatsiya)

In [ ]:
last = pipeline.getStages()[-1]
if is_linear_reg(VARIANT):
    grid = ParamGridBuilder().addGrid(last.regParam, [0.01, 0.1]).addGrid(last.elasticNetParam, [0.0, 0.5]).build()
elif VARIANT == 2:
    grid = ParamGridBuilder().addGrid(last.numTrees, [20, 40]).addGrid(last.maxDepth, [5, 7]).build()
else:
    grid = ParamGridBuilder().addGrid(last.maxDepth, [3, 5]).addGrid(last.maxIter, [20]).build()

evaluator = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="rmse")
cv = CrossValidator(estimator=pipeline, estimatorParamMaps=grid, evaluator=evaluator, numFolds=3)
cv_model = cv.fit(train_df)

#### Otsenka na testovoy vyborke

In [ ]:
pred = cv_model.transform(test_df)
rmse = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="rmse").evaluate(pred)
r2 = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="r2").evaluate(pred)
print(f"RMSE: {rmse:.3f}")
print(f"R2: {r2:.3f}")
pred.select(*[col_n for col_n in pred.columns if col_n != target_col], target_col).show(5)

#### Vyvody
Opishi kachestvo modeli regressii (RMSE, R2), vliyanie priznakov i giperparametrov.